In [2]:
import pandas as pd
import numpy as np
import json
from math import comb


with open("winrates.json") as f:
    winrates = json.load(f)

decks = list(winrates.keys())

def get_matchup_wr(deck, opponent, winrates=winrates):
    if deck == opponent:
        return 0.5
    return winrates[deck]["vs"].get(opponent, {}).get("winrate", 0.5)


def get_meta_from_mtgdecks(additional_decks=None):
    total_matches = sum(winrates[d]["overall"]["matches"] for d in decks)
    meta_share = {
        d: winrates[d]["overall"]["matches"] / total_matches
        for d in decks
    }
    if additional_decks:
        for deck in additional_decks:
            if deck in meta_share:
                meta_share[deck] += 1
            else:
                meta_share[deck] = 1
        total = sum(meta_share.values())
        meta_share = {d: meta_share[d] / total for d in meta_share}
    return meta_share


def get_general_winrates(meta_share=None):
    if meta_share is None:
        return {
            d: winrates[d]["overall"]["winrate"]
            for d in decks
        }
    return {
        d: sum(
            get_matchup_wr(d, opp) * meta_share[opp]
            for opp in decks
        )
        for d in decks
    }

def get_meta_from_file(filename="metagame.txt", additional_decks=None):
    meta_share = {}
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            deck = line.strip()

            if not deck:
                continue

            if deck not in meta_share:
                meta_share[deck] = 1
            else:
                meta_share[deck] += 1
    for deck in additional_decks or []:
        if deck not in meta_share:
            meta_share[deck] = 1
        else:
            meta_share[deck] += 1
    meta_share = {d: meta_share.get(d, 0) for d in decks}
    total = sum(meta_share.values())
    meta_share = {d: meta_share[d] / total for d in decks}
    return meta_share



def probabilistic_tournament(meta_share, n_rounds=5):
    decks_performance = {(0, 0): {d: meta_share[d] for d in decks}}

    for _ in range(n_rounds):
        new_decks_performance = {}

        bracket_fields = {}
        for result, dist in decks_performance.items():
            # total = sum(dist.values())
            # if total > 0:
            bracket_fields[result] = {d: dist[d] for d in decks}

        for result, dist in decks_performance.items():
            wins, losses = result
            win_key = (wins + 1, losses)
            loss_key = (wins, losses + 1)

            if win_key not in new_decks_performance:
                new_decks_performance[win_key] = {d: 0 for d in decks}
            if loss_key not in new_decks_performance:
                new_decks_performance[loss_key] = {d: 0 for d in decks}

            # Normalize a copy — never mutate dist
            total = sum(dist.values())
            field = {d: dist[d] / total for d in decks} if total > 0 else {d: 0 for d in decks}

            for deck in decks:
                for opp in decks:
                    matchup_wr = get_matchup_wr(deck, opp)
                    weight = dist[deck] * field[opp]

                    new_decks_performance[win_key][deck] += weight * matchup_wr
                    new_decks_performance[loss_key][deck] += weight * (1 - matchup_wr)
        decks_performance = new_decks_performance

    return decks_performance

In [3]:
def summarize_tournament(meta_share=None, n_rounds=5):
    if meta_share is None:
        meta_share = get_meta_from_file()
    outcomes = probabilistic_tournament(meta_share, n_rounds=n_rounds)
    rows = []
    general_wr = get_general_winrates(meta_share)

    for deck in decks:
        if meta_share[deck] == 0:
            continue
        
        row = {
            "Deck": deck,
            "Meta Share": meta_share[deck],
            "Winrate vs Meta": general_wr[deck],
        }
        
        for result, dist in sorted(outcomes.items()):
            wins, losses = result
            total = sum(dist.values())
            deck_prob_in_bracket = dist[deck] / total if total > 0 else 0
            row[f"P({wins}W-{losses}L)"] = dist[deck]
            row[f"Field%@{wins}W-{losses}L"] = deck_prob_in_bracket
        
        rows.append(row)
        row["Delta"] = row["Field%@5W-0L"] / row["Meta Share"] if row["Meta Share"] > 0 else np.nan

    df = pd.DataFrame(rows).sort_values("Meta Share", ascending=False)

    base_cols = ["Deck", "Meta Share", "Winrate vs Meta"]
    bracket_prob_cols = sorted([c for c in df.columns if c.startswith("P(")], 
                                key=lambda x: tuple(int(n) for n in x.replace("P(","").replace(")","").replace("W-","_").replace("L","").split("_")))
    field_cols = sorted([c for c in df.columns if c.startswith("Field%")],
                        key=lambda x: tuple(int(n) for n in x.replace("Field%@","").replace("W-","_").replace("L","").split("_")))

    delta = ["Delta"]

    df = df[base_cols + bracket_prob_cols[-3:] + field_cols[-3:] + delta]
    display(df)

In [4]:
def test_deck_choices(n_rounds=5):
    tested_decks = dict.fromkeys(sorted(decks, key=lambda d: winrates[d]["overall"]["matches"], reverse=True)[:30], 0)

    for d in tested_decks:
        meta_share_with_deck = get_meta_from_file(filename="meta-spring-2026-3.txt", additional_decks=[d])
        # meta_share_with_deck = get_meta_from_mtgdecks(additional_decks=[d])
        general_wr = get_general_winrates(meta_share_with_deck)
        outcomes = probabilistic_tournament(meta_share_with_deck, n_rounds=n_rounds)
        deck_performance = {result: dist[d] for result, dist in outcomes.items()}
        total = sum(deck_performance.values())
        deck_performance = {result: v / total for result, v in deck_performance.items()}
        sorted_results = sorted(deck_performance.keys())
        tested_decks[d] = {result: deck_performance[result] for result in sorted_results}

    tested_decks_df = pd.DataFrame(tested_decks).T
    tested_decks_df.insert(0, "Win%", [general_wr[d] for d in tested_decks_df.index])
    tested_decks_df = tested_decks_df.sort_values((5, 0), ascending=False)
    display(tested_decks_df)

test_deck_choices()

,Win%,0,1,2,3,4,5
,,5,4,3,2,1,0
Jeskai Ephemerate,0.595833,0.008051,0.063338,0.212937,0.358594,0.279719,0.077361
Spy Combo,0.614167,0.006405,0.065244,0.233754,0.368879,0.259794,0.065924
GW Bogles,0.577500,0.019916,0.111737,0.262617,0.328784,0.217672,0.059274
Red Storm,0.557917,0.016592,0.108210,0.269885,0.334730,0.213889,0.056694
UWx Familiar,0.590833,0.014398,0.093917,0.258098,0.353306,0.226955,0.053325
Elves,0.562083,0.013805,0.099542,0.271610,0.350055,0.214634,0.050354
Altar Tron,0.602500,0.006771,0.074710,0.268330,0.383499,0.222986,0.043705
Mono Black Sacrifice,0.535417,0.021447,0.120790,0.287353,0.341158,0.190457,0.038795
Dimir Terror,0.508333,0.024736,0.140926,0.307144,0.323154,0.168432,0.035607


In [ ]:
import numpy as np
from scipy import stats


def get_tournament_winrates(filename: str, n_rounds: int = 5) -> dict:
    """
    Odczytuje plik z deckami (jeden per linia, posortowane od 1. miejsca),
    przypisuje winrate na podstawie rozkładu normalnego przeskalowanego do N graczy.
    Używa rozkładu normalnego jako aproksymacji binomialnej (p=0.5, n=n_rounds).
    """
    with open(filename, "r", encoding="utf-8") as f:
        decks_in_order = [line.strip() for line in f.read().splitlines() if line.strip()]

    n_players = len(decks_in_order)
    possible_wins = list(range(n_rounds + 1))  # [0, 1, 2, 3, 4, 5]
    mean = n_rounds / 2          # 2.5
    std = np.sqrt(n_rounds * 0.25)  # sqrt(5 * 0.25) ≈ 1.118

    # P(wins = w) z rozkładu normalnego (aproksymacja binomialnej)
    raw = {w: stats.norm.pdf(w, loc=mean, scale=std) for w in possible_wins}
    total_raw = sum(raw.values())
    counts = {w: round(v / total_raw * n_players) for w, v in raw.items()}

    # Korekta błędów zaokrągleń – dodaj/odejmij od środkowego wyniku
    diff = n_players - sum(counts.values())
    counts[n_rounds // 2] += diff

    print(f"\nRozkład graczy ({n_players} total):")
    print(f"{'Wynik':<10} {'Liczba graczy':<15} {'WR%'}")
    for w in sorted(possible_wins, reverse=True):
        print(f"{w}-{n_rounds - w:<8} {counts[w]:<15} {w / n_rounds * 100:.1f}%")

    # Przypisz WR każdemu deckowi według miejsca (rank 0 = najlepszy)
    deck_winrates = {}
    rank = 0
    for w in sorted(possible_wins, reverse=True):
        for _ in range(counts[w]):
            if rank < n_players:
                deck_winrates[decks_in_order[rank]] = w / n_rounds
                rank += 1

    return deck_winrates


def compare_tournament_vs_model(filename: str, n_rounds: int = 5):
    """
    Porównuje winrate z turnieju (przypisany przez rozkład normalny)
    z oczekiwanym winrate wyliczonym przez probabilistic_tournament.
    """
    tournament_wr = get_tournament_winrates(filename, n_rounds)

    meta_share = get_meta_from_file(filename=filename)
    outcomes = probabilistic_tournament(meta_share, n_rounds=n_rounds)

    # Wylicz E[wins] / n_rounds dla każdego decku z rozkładu outcomes
    # outcomes: {(wins, losses): {deck: masa_prawdopodobienstwa}}
    def model_expected_wr(deck):
        total_mass = 0.0
        weighted_wins = 0.0
        for (wins, losses), dist in outcomes.items():
            mass = dist.get(deck, 0)
            weighted_wins += wins * mass
            total_mass += mass
        if total_mass == 0:
            return None
        return (weighted_wins / total_mass) / n_rounds

    # Zbuduj DataFrame
    rows = []
    for deck, t_wr in tournament_wr.items():
        m_wr = model_expected_wr(deck)
        rows.append({
            "Deck": deck,
            "Tournament WR%": round(t_wr * 100, 1),
            "Model WR%": round(m_wr * 100, 1) if m_wr is not None else None,
        })

    df = pd.DataFrame(rows).set_index("Deck")
    df["Diff"] = df["Model WR%"] - df["Tournament WR%"]
    df = df.sort_values("Tournament WR%", ascending=False)

    # Korelacje
    valid = df.dropna()
    if len(valid) >= 2:
        pearson_r, pearson_p = stats.pearsonr(valid["Tournament WR%"], valid["Model WR%"])
        spearman_r, spearman_p = stats.spearmanr(valid["Tournament WR%"], valid["Model WR%"])
        print(f"\nKorelacja (n={len(valid)} decków):")
        print(f"  Pearson r  = {pearson_r:+.3f}  (p={pearson_p:.3f})")
        print(f"  Spearman r = {spearman_r:+.3f}  (p={spearman_p:.3f})")
        mae = valid["Diff"].abs().mean()
        print(f"  MAE        = {mae:.2f}pp")

    display(df)
    return df


compare_tournament_vs_model("meta-spring-2026-3.txt")


Rozkład graczy (25 total):
Wynik      Liczba graczy   WR%
5-0        1               100.0%
4-1        4               80.0%
3-2        8               60.0%
2-3        7               40.0%
1-4        4               20.0%
0-5        1               0.0%


,Tournament WR%,Model WR%
Deck,,
Rakdos Madness,80.0,None
Goblin Combo (MoggWarts),80.0,None
Jund Wildfire,60.0,None
Ruby Storm,60.0,None
Spy Combo,60.0,None
Gruul Ramp,60.0,None
Esper Affinity,40.0,None
Grixis Affinity,40.0,None
Caw Gates,40.0,None


,Tournament WR%,Model WR%
Deck,,
Rakdos Madness,80.0,None
Goblin Combo (MoggWarts),80.0,None
Jund Wildfire,60.0,None
Ruby Storm,60.0,None
Spy Combo,60.0,None
Gruul Ramp,60.0,None
Esper Affinity,40.0,None
Grixis Affinity,40.0,None
Caw Gates,40.0,None
